In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [14]:
DATA_FOLDER = "preprocessed_avoiding"

# Select only relevant trials from whole data set

In [15]:
class Condition(pd.DataFrame):

    def __init__(self, file: str, condition:str):
        super().__init__(pd.read_csv(file))
        self.condition = condition

    def get_condition(self):
        return self.condition

In [16]:
ai = Condition(f"{DATA_FOLDER}/ai.csv", "ai")
ad = Condition(f"{DATA_FOLDER}/ad.csv", "ad")

In [17]:
def get_relevant_row(df: pd.DataFrame):
    """Gets the actual row per trial"""
    type = df.loc[:, "task"].iloc[0]
    if type == "avoiding":
        switched = df.loc[df["switched_side"] == 1]
        return switched.iloc[0] if len(switched) > 0 else df.iloc[-1]
    elif type == "reaching":
        return df.iloc[-1]
    else:
        raise ValueError(f"Faulty trial type found! No typ of {type}")

def find_relevant_entry(df: pd.DataFrame):
    """Separates the phase df into blocks and returns one row per block"""
    #print(df)
    return df.groupby(["trial_index"]).apply(get_relevant_row, include_groups=False)

In [18]:
for cond in [ai, ad]:
    nested_group = cond.groupby(["participant_id", "phase", "block"])
    grouped = nested_group.apply(find_relevant_entry, include_groups=False)
    grouped.to_csv(f"{DATA_FOLDER}/{cond.get_condition()}_relevant_trials.csv")

In [19]:
del ai
del ad

# Calculate mean and variance per block

In [20]:
ai = Condition(f"{DATA_FOLDER}/ai_relevant_trials.csv", "ai")
ad = Condition(f"{DATA_FOLDER}/ad_relevant_trials.csv", "ad")

In [23]:
def get_mean_variance_median_per_block(df: pd.DataFrame):
    df["difference"] = df.loc[:,"target_pos_y"] - df.loc[:,"current_pos_y_cm"]
    return pd.DataFrame({
        "diff_mean": [df["difference"].mean()],
        "diff_variance": [df["difference"].var()],
        "mapping": [df["mapping"].iloc[0]],
        "natural_mapping": [df["natural_mapping"].iloc[0]]
    })

In [24]:
for cond in [ai, ad]:
    grouped = cond.groupby(["participant_id", "phase", "block", "task"]).apply(get_mean_variance_median_per_block, include_groups=False)
    grouped.to_csv(f"{DATA_FOLDER}/{cond.get_condition()}_block_measures.csv")